# E04 – PyTorch alapok 2
**Generatív AI és inverz módszerek a képszintézisben** | BME, 2026

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/VaitkusM/genai-inverse-graphics-bme-vik-iit/blob/main/eloadasok/E04_PyTorch-2_NN.ipynb)

## Tartalom
1. MNIST osztályozás MLP-vel
2. CIFAR-10 osztályozás konvolúciós hálóval
3. U-Net denoizer CIFAR-10-en

In [ ]:
# Colab-on szükséges csomagok telepítése (lokálisan általában nem kell)
import sys
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    !pip install torch torchvision matplotlib numpy --quiet


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader, TensorDataset
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 100

# Eszköz kiválasztása: GPU ha elérhető, egyébként CPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Használt eszköz: {device}')
print(f'PyTorch verzió: {torch.__version__}')


---
## 1. rész – MNIST osztályozás MLP-vel
Az MNIST adatbázis 28×28 pixeles, fekete-fehér kézzel írt számjegyeket tartalmaz (0–9).
Egy egyszerű **teljesen összekötött hálót** (MLP) tanítunk rá.

In [ ]:
# MNIST betöltése torchvision-nal
mnist_transform = transforms.Compose([
    transforms.ToTensor(),                        # [0,255] -> [0,1]
    transforms.Normalize((0.1307,), (0.3081,))   # MNIST átlag/szórás
])

mnist_train = torchvision.datasets.MNIST('./data', train=True,  download=True, transform=mnist_transform)
mnist_test  = torchvision.datasets.MNIST('./data', train=False, download=True, transform=mnist_transform)

train_loader_mnist = DataLoader(mnist_train, batch_size=64, shuffle=True,  num_workers=0)
test_loader_mnist  = DataLoader(mnist_test,  batch_size=256, shuffle=False, num_workers=0)

print(f'Tanítóhalmaz mérete: {len(mnist_train)}')
print(f'Teszthalmaz mérete:  {len(mnist_test)}')

# Néhány példa megjelenítése
kepek, cimkek = next(iter(train_loader_mnist))
fig, axes = plt.subplots(2, 8, figsize=(12, 4.5))
for i, ax in enumerate(axes.flat):
    ax.imshow(kepek[i].squeeze(), cmap='gray')
    ax.set_title(str(cimkek[i].item()), fontsize=10)
    ax.axis('off')
plt.suptitle('MNIST minták', fontsize=12)
plt.tight_layout(); plt.show()


### 1.1 MLP modell definíciója
Felépítés: **784 → 256 → 128 → 10** (bemenet: kiterített 28×28 kép, kimenet: 10 osztály)

In [ ]:
class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Flatten(),               # 28x28 -> 784
            nn.Linear(784, 256),
            nn.ReLU(),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Linear(128, 10)          # logitok (CrossEntropyLoss-hoz nem kell Softmax)
        )

    def forward(self, x):
        return self.net(x)


mlp = MLP().to(device)
print(mlp)
print(f'\nParaméterek száma: {sum(p.numel() for p in mlp.parameters()):,}')


### 1.2 Tanítási hurok

In [ ]:
def tanit_egy_epoch(model, loader, criterion, optimizer, device):
    """Egy epoch tanítás, visszaadja az átlagos veszteséget és pontosságot."""
    model.train()
    ossz_veszteség = 0.0
    helyes = 0
    ossz = 0

    for kepek, cimkek in loader:
        kepek, cimkek = kepek.to(device), cimkek.to(device)

        optimizer.zero_grad()           # gradiensek nullázása
        kimenetek = model(kepek)        # előrevetítés
        loss = criterion(kimenetek, cimkek)  # veszteség
        loss.backward()                 # visszaterjesztés
        optimizer.step()                # paraméter frissítés

        ossz_veszteség += loss.item() * kepek.size(0)
        _, josolt = kimenetek.max(1)
        helyes += josolt.eq(cimkek).sum().item()
        ossz   += kepek.size(0)

    return ossz_veszteség / ossz, 100. * helyes / ossz


def kiertekkel(model, loader, criterion, device):
    """Kiértékelés (nincs gradiens-számítás)."""
    model.eval()
    ossz_veszteség = 0.0
    helyes = 0
    ossz = 0

    with torch.no_grad():
        for kepek, cimkek in loader:
            kepek, cimkek = kepek.to(device), cimkek.to(device)
            kimenetek = model(kepek)
            loss = criterion(kimenetek, cimkek)
            ossz_veszteség += loss.item() * kepek.size(0)
            _, josolt = kimenetek.max(1)
            helyes += josolt.eq(cimkek).sum().item()
            ossz   += kepek.size(0)

    return ossz_veszteség / ossz, 100. * helyes / ossz


In [ ]:
# Tanítás
mlp_criterion = nn.CrossEntropyLoss()
mlp_optimizer = optim.Adam(mlp.parameters(), lr=1e-3)

N_EPOCH_MNIST = 5
history_mnist = {'train_loss': [], 'train_acc': [], 'test_loss': [], 'test_acc': []}

print('Epoch | Train Loss | Train Acc | Test Loss | Test Acc')
print('-' * 55)
for epoch in range(1, N_EPOCH_MNIST + 1):
    tr_loss, tr_acc = tanit_egy_epoch(mlp, train_loader_mnist, mlp_criterion, mlp_optimizer, device)
    te_loss, te_acc = kiertekkel(mlp, test_loader_mnist, mlp_criterion, device)

    history_mnist['train_loss'].append(tr_loss)
    history_mnist['train_acc'].append(tr_acc)
    history_mnist['test_loss'].append(te_loss)
    history_mnist['test_acc'].append(te_acc)

    print(f'  {epoch:3d}  | {tr_loss:.4f}     | {tr_acc:.2f}%   | {te_loss:.4f}    | {te_acc:.2f}%')


In [ ]:
# Veszteség és pontosság görbe
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))
epochok = range(1, N_EPOCH_MNIST + 1)

ax1.plot(epochok, history_mnist['train_loss'], label='Tanítás')
ax1.plot(epochok, history_mnist['test_loss'],  label='Teszt')
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Veszteség')
ax1.set_title('MNIST – Veszteség'); ax1.legend()

ax2.plot(epochok, history_mnist['train_acc'], label='Tanítás')
ax2.plot(epochok, history_mnist['test_acc'],  label='Teszt')
ax2.set_xlabel('Epoch'); ax2.set_ylabel('Pontosság (%)')
ax2.set_title('MNIST – Pontosság'); ax2.legend()

plt.tight_layout(); plt.show()


In [ ]:
# Néhány jóslat megjelenítése
mlp.eval()
kepek, cimkek = next(iter(test_loader_mnist))
kepek_gpu = kepek.to(device)

with torch.no_grad():
    kimenetek = mlp(kepek_gpu)
    _, josolt = kimenetek.max(1)

fig, axes = plt.subplots(2, 8, figsize=(12, 4.5))
for i, ax in enumerate(axes.flat):
    ax.imshow(kepek[i].squeeze(), cmap='gray')
    helyes_e = josolt[i].item() == cimkek[i].item()
    szin = 'green' if helyes_e else 'red'
    ax.set_title(f'J:{josolt[i].item()} V:{cimkek[i].item()}', color=szin, fontsize=8)
    ax.axis('off')
plt.suptitle('MLP jóslatok (J=Jósolt, V=Valódi, zöld=helyes, piros=hibás)', fontsize=10)
plt.tight_layout(); plt.show()


---
## 2. rész – CIFAR-10 osztályozás konvolúciós hálóval (CNN)
A CIFAR-10 adatbázis 32×32 pixeles, 10 osztályba sorolt színes képeket tartalmaz.
Egy **konvolúciós neurális hálót** (CNN) definiálunk és tanítunk rá.

In [ ]:
# CIFAR-10 osztályok nevei
CIFAR10_OSZTALYOK = ['repülő', 'autó', 'madár', 'macska', 'szarvas',
                     'kutya', 'béka', 'ló', 'hajó', 'tehergépkocsi']

# Adataugmentáció a tanítóhalmazhoz, normalizálás mindkettőhöz
cifar_train_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(),             # véletlen vízszintes tükrözés
    transforms.RandomCrop(32, padding=4),          # véletlen kivágás
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465),  # CIFAR-10 csatorna-átlagok
                         (0.2023, 0.1994, 0.2010))  # CIFAR-10 csatorna-szórások
])

cifar_test_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465),
                         (0.2023, 0.1994, 0.2010))
])

cifar_train = torchvision.datasets.CIFAR10('./data', train=True,  download=True, transform=cifar_train_transform)
cifar_test  = torchvision.datasets.CIFAR10('./data', train=False, download=True, transform=cifar_test_transform)

train_loader_cifar = DataLoader(cifar_train, batch_size=128, shuffle=True,  num_workers=0, pin_memory=True)
test_loader_cifar  = DataLoader(cifar_test,  batch_size=256, shuffle=False, num_workers=0, pin_memory=True)

print(f'Tanítóhalmaz: {len(cifar_train)} kép')
print(f'Teszthalmaz:  {len(cifar_test)}  kép')


In [ ]:
# Néhány CIFAR-10 minta megjelenítése (denormalizálva)
def denorm(x, mean=(0.4914, 0.4822, 0.4465), std=(0.2023, 0.1994, 0.2010)):
    m = torch.tensor(mean).view(3, 1, 1)
    s = torch.tensor(std).view(3, 1, 1)
    return (x * s + m).clamp(0, 1)

kepek_c, cimkek_c = next(iter(train_loader_cifar))
fig, axes = plt.subplots(2, 8, figsize=(12, 4))
for i, ax in enumerate(axes.flat):
    img = denorm(kepek_c[i]).permute(1, 2, 0).numpy()
    ax.imshow(img)
    ax.set_title(CIFAR10_OSZTALYOK[cimkek_c[i].item()], fontsize=8)
    ax.axis('off')
plt.suptitle('CIFAR-10 minták', fontsize=12)
plt.tight_layout(); plt.show()


### 2.1 Konvolúciós háló definíciója
Felépítés: **2× (Conv-BN-ReLU-MaxPool)** blokk + fully-connected osztályozó fej.

In [ ]:
class ConvNet(nn.Module):
    def __init__(self, n_osztaly=10):
        super().__init__()

        # 1. konvolúciós blokk: 3 csatorna -> 32
        self.blokk1 = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),  # 32x32 -> 32x32
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2)                              # 32x32 -> 16x16
        )

        # 2. konvolúciós blokk: 32 -> 64 csatorna
        self.blokk2 = nn.Sequential(
            nn.Conv2d(32, 64, kernel_size=3, padding=1),  # 16x16 -> 16x16
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2)                               # 16x16 -> 8x8
        )

        # Osztályozó fej
        self.fej = nn.Sequential(
            nn.Flatten(),           # 64 * 8 * 8 = 4096
            nn.Linear(64 * 8 * 8, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, n_osztaly)
        )

    def forward(self, x):
        x = self.blokk1(x)   # (B, 32, 16, 16)
        x = self.blokk2(x)   # (B, 64, 8, 8)
        x = self.fej(x)      # (B, 10)
        return x


cnn = ConvNet().to(device)
print(cnn)
print(f'\nParaméterek száma: {sum(p.numel() for p in cnn.parameters()):,}')


### 2.2 Tanítási hurok

In [ ]:
cnn_criterion = nn.CrossEntropyLoss()
cnn_optimizer = optim.Adam(cnn.parameters(), lr=1e-3)
# Tanulási ráta csökkentése 10 epoch után
cnn_scheduler = optim.lr_scheduler.StepLR(cnn_optimizer, step_size=10, gamma=0.3)

N_EPOCH_CIFAR = 15
history_cifar = {'train_loss': [], 'train_acc': [], 'test_loss': [], 'test_acc': []}

print('Epoch | Train Loss | Train Acc | Test Loss | Test Acc')
print('-' * 55)
for epoch in range(1, N_EPOCH_CIFAR + 1):
    tr_loss, tr_acc = tanit_egy_epoch(cnn, train_loader_cifar, cnn_criterion, cnn_optimizer, device)
    te_loss, te_acc = kiertekkel(cnn, test_loader_cifar, cnn_criterion, device)
    cnn_scheduler.step()

    history_cifar['train_loss'].append(tr_loss)
    history_cifar['train_acc'].append(tr_acc)
    history_cifar['test_loss'].append(te_loss)
    history_cifar['test_acc'].append(te_acc)

    print(f'  {epoch:3d}  | {tr_loss:.4f}     | {tr_acc:.2f}%   | {te_loss:.4f}    | {te_acc:.2f}%')


In [ ]:
# Veszteség és pontosság görbe
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))
epochok = range(1, N_EPOCH_CIFAR + 1)

ax1.plot(epochok, history_cifar['train_loss'], label='Tanítás')
ax1.plot(epochok, history_cifar['test_loss'],  label='Teszt')
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Veszteség')
ax1.set_title('CIFAR-10 – Veszteség'); ax1.legend()

ax2.plot(epochok, history_cifar['train_acc'], label='Tanítás')
ax2.plot(epochok, history_cifar['test_acc'],  label='Teszt')
ax2.set_xlabel('Epoch'); ax2.set_ylabel('Pontosság (%)')
ax2.set_title('CIFAR-10 – Pontosság'); ax2.legend()

plt.tight_layout(); plt.show()


In [ ]:
# Jóslatok megjelenítése a teszthalmazon
cnn.eval()
kepek_t, cimkek_t = next(iter(test_loader_cifar))
kepek_t_gpu = kepek_t.to(device)

with torch.no_grad():
    kimenetek_t = cnn(kepek_t_gpu)
    _, josolt_t = kimenetek_t.max(1)

fig, axes = plt.subplots(2, 8, figsize=(14, 4))
for i, ax in enumerate(axes.flat):
    img = denorm(kepek_t[i]).permute(1, 2, 0).numpy()
    ax.imshow(img)
    helyes_e = josolt_t[i].item() == cimkek_t[i].item()
    szin = 'green' if helyes_e else 'red'
    ax.set_title(f'{CIFAR10_OSZTALYOK[josolt_t[i].item()]}\n({CIFAR10_OSZTALYOK[cimkek_t[i].item()]})',
                 color=szin, fontsize=7)
    ax.axis('off')
plt.suptitle('CNN jóslatok (felső: jósolt, zárójelben: valódi; zöld=helyes, piros=hibás)', fontsize=10)
plt.tight_layout(); plt.show()


---
## 3. rész – U-Net denoizer CIFAR-10-en

A **denoizing** (zajeltávolítás) feladata: adott egy zajos kép, a cél a tiszta kép visszaállítása.

**Megközelítés:**
- Gauss-zajt adunk a normalizált CIFAR-10 képekhez
- Egy **U-Net** architektúrájú hálót tanítunk: bemenet = zajos kép, célérték = tiszta kép
- Veszteségfüggvény: MSE (Mean Squared Error)

**U-Net felépítése (32×32-es képekre):**
- Enkóder: 3 × (DuplKonv → MaxPool), csatornaszámok: 3→32→64→128
- Palack-nyak (bottleneck): DuplKonv 128→256
- Dekóder: 3 × (Upsample + skip connection → DuplKonv)
- Kimenet: 1×1 Conv → 3 csatorna (RGB)

A **skip connection**-ök az enkóder és dekóder azonos felbontású szintjeit kötik össze,
megőrizve a finom térbeli részleteket.

In [ ]:
# Az U-Net alapegysége: két egymás utáni konvolúciós réteg
class DuplKonv(nn.Module):
    def __init__(self, be_cs, ki_cs):
        super().__init__()
        self.blokk = nn.Sequential(
            nn.Conv2d(be_cs, ki_cs, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(ki_cs),
            nn.ReLU(inplace=True),
            nn.Conv2d(ki_cs, ki_cs, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(ki_cs),
            nn.ReLU(inplace=True),
        )
    def forward(self, x):
        return self.blokk(x)


class UNet(nn.Module):
    def __init__(self, be_csatorna=3, ki_csatorna=3, alap=32):
        super().__init__()
        # Enkóder
        self.enc1 = DuplKonv(be_csatorna, alap)       # (B,  3, 32,32) -> (B, 32, 32,32)
        self.enc2 = DuplKonv(alap,     alap * 2)       # (B, 32, 16,16) -> (B, 64, 16,16)
        self.enc3 = DuplKonv(alap * 2, alap * 4)       # (B, 64,  8, 8) -> (B,128,  8, 8)
        self.pool = nn.MaxPool2d(2)                    # minden enkóder után

        # Palack-nyak
        self.bottleneck = DuplKonv(alap * 4, alap * 8) # (B,128, 4, 4) -> (B,256, 4, 4)

        # Dekóder: transzponált konv (felskálázás) + skip + DuplKonv
        self.up3   = nn.ConvTranspose2d(alap * 8, alap * 4, kernel_size=2, stride=2)
        self.dec3  = DuplKonv(alap * 8, alap * 4)     # concat: 256 -> 128

        self.up2   = nn.ConvTranspose2d(alap * 4, alap * 2, kernel_size=2, stride=2)
        self.dec2  = DuplKonv(alap * 4, alap * 2)     # concat: 128 -> 64

        self.up1   = nn.ConvTranspose2d(alap * 2, alap, kernel_size=2, stride=2)
        self.dec1  = DuplKonv(alap * 2, alap)         # concat:  64 -> 32

        # 1×1 konvolúció: csatornaszám visszaállítása
        self.kimenet = nn.Conv2d(alap, ki_csatorna, kernel_size=1)

    def forward(self, x):
        # Enkóder + skip connectionök elmentése
        e1 = self.enc1(x)                   # (B, 32, 32, 32)
        e2 = self.enc2(self.pool(e1))        # (B, 64, 16, 16)
        e3 = self.enc3(self.pool(e2))        # (B,128,  8,  8)

        # Palack-nyak
        b = self.bottleneck(self.pool(e3))   # (B,256,  4,  4)

        # Dekóder: felskálázás + skip connection összefűzése (torch.cat) + DuplKonv
        d3 = self.dec3(torch.cat([self.up3(b),  e3], dim=1))  # (B,128, 8, 8)
        d2 = self.dec2(torch.cat([self.up2(d3), e2], dim=1))  # (B, 64,16,16)
        d1 = self.dec1(torch.cat([self.up1(d2), e1], dim=1))  # (B, 32,32,32)

        return self.kimenet(d1)              # (B,  3, 32, 32)


unet = UNet().to(device)
print(f'U-Net paraméterek száma: {sum(p.numel() for p in unet.parameters()):,}')
# Gyors formai ellenőrzés
_x = torch.randn(2, 3, 32, 32, device=device)
assert unet(_x).shape == (2, 3, 32, 32), 'Helytelen kimeneti forma!'
print('Formai ellenőrzés: OK  (bemenet: (2,3,32,32)  →  kimenet: (2,3,32,32))')

In [ ]:
ZAJ_SZINT = 0.3  # Gauss-zaj szórása (normalizált képtérben)

def zajt_ad(x, szint=ZAJ_SZINT):
    """Gauss-zajt ad normalizált képekhez (in-place klónozás nélkül)."""
    return x + torch.randn_like(x) * szint

# Néhány tiszta–zajos pár megjelenítése
kepek_demo, _ = next(iter(test_loader_cifar))
zajos_demo = zajt_ad(kepek_demo[:8])

fig, axes = plt.subplots(2, 8, figsize=(14, 4))
for i in range(8):
    axes[0, i].imshow(denorm(kepek_demo[i]).permute(1, 2, 0).clamp(0,1).numpy())
    axes[1, i].imshow(denorm(zajos_demo[i]).permute(1, 2, 0).clamp(0,1).numpy())
    axes[0, i].axis('off')
    axes[1, i].axis('off')
axes[0, 0].set_ylabel('Tiszta', fontsize=10)
axes[1, 0].set_ylabel('Zajos', fontsize=10)
plt.suptitle(f'Zajhozzáadás (σ = {ZAJ_SZINT}, normalizált képtérben)', fontsize=11)
plt.tight_layout(); plt.show()

### 3.1 Tanítási hurok

A denoizer tanításának lépései minden batch-re:
1. Zajos kép generálása: `zajos = tiszta + N(0, σ²)`
2. Forward pass: `tisztított = unet(zajos)`
3. MSE veszteség a tiszta képpel szemben: `loss = MSE(tisztított, tiszta)`
4. Backpropagation + optimalizáló lépés

In [ ]:
def denoizer_epoch(model, loader, optimizer, device, zaj_szint=ZAJ_SZINT):
    """Egy epoch tanítás a denoizer hálónak.
    
    Visszatér: átlagos MSE veszteség az epochon.
    """
    model.train()
    ossz_loss = 0.0

    for kepek, _ in loader:               # osztálycímkék nem kellenek
        kepek = kepek.to(device)
        zajos = zajt_ad(kepek, zaj_szint) # zaj hozzáadása

        optimizer.zero_grad()
        tisztitott = model(zajos)         # forward pass
        loss = nn.functional.mse_loss(tisztitott, kepek)  # MSE a tiszta képpel
        loss.backward()
        optimizer.step()

        ossz_loss += loss.item() * kepek.size(0)

    return ossz_loss / len(loader.dataset)


@torch.no_grad()
def denoizer_kiert(model, loader, device, zaj_szint=ZAJ_SZINT):
    """Kiértékelés a teszthalmazon (gradiens nélkül)."""
    model.eval()
    ossz_loss = 0.0
    for kepek, _ in loader:
        kepek = kepek.to(device)
        zajos = zajt_ad(kepek, zaj_szint)
        tisztitott = model(zajos)
        ossz_loss += nn.functional.mse_loss(tisztitott, kepek).item() * kepek.size(0)
    return ossz_loss / len(loader.dataset)

In [ ]:
unet_optimizer = optim.Adam(unet.parameters(), lr=1e-3)
unet_scheduler = optim.lr_scheduler.StepLR(unet_optimizer, step_size=8, gamma=0.3)

N_EPOCH_UNET = 15
history_unet = {'train': [], 'test': []}

print('Epoch | Train MSE  | Test MSE')
print('-' * 32)
for epoch in range(1, N_EPOCH_UNET + 1):
    tr = denoizer_epoch(unet, train_loader_cifar, unet_optimizer, device)
    te = denoizer_kiert(unet, test_loader_cifar,  device)
    unet_scheduler.step()
    history_unet['train'].append(tr)
    history_unet['test'].append(te)
    print(f'  {epoch:3d}  | {tr:.5f}    | {te:.5f}')

In [ ]:
# Veszteség görbe
plt.figure(figsize=(7, 3))
plt.plot(history_unet['train'], label='Tanítás')
plt.plot(history_unet['test'],  label='Teszt')
plt.xlabel('Epoch'); plt.ylabel('MSE veszteség')
plt.title('U-Net denoizer – tanítási görbe')
plt.legend(); plt.tight_layout(); plt.show()

# Vizuális eredmény: tiszta – zajos – denoizált
unet.eval()
kepek_vis, _ = next(iter(test_loader_cifar))
kepek_vis = kepek_vis[:8]
zajos_vis  = zajt_ad(kepek_vis)

with torch.no_grad():
    tisztitott_vis = unet(zajos_vis.to(device)).cpu()

fig, axes = plt.subplots(3, 8, figsize=(14, 6))
sorok = ['Tiszta', 'Zajos', 'Denoizált']
for i in range(8):
    for sor, img in enumerate([kepek_vis[i], zajos_vis[i], tisztitott_vis[i]]):
        axes[sor, i].imshow(denorm(img).permute(1, 2, 0).clamp(0, 1).numpy())
        axes[sor, i].axis('off')
for sor, nev in enumerate(sorok):
    axes[sor, 0].set_ylabel(nev, fontsize=10)
plt.suptitle('U-Net denoizer eredmények', fontsize=12)
plt.tight_layout(); plt.show()